# TAQ Matching — viz & test

In [6]:
import os
import sys

os.environ["QUANTDATA_CACHE_MODE"] = "off"

%load_ext autoreload
%autoreload 1
project_root = os.path.abspath("/Users/elie/Documents/GitHub/quant_data/")

if project_root not in sys.path:
    sys.path.insert(0, project_root)
%aimport quantdata
%aimport quantdata.biz_calendar
%aimport quantdata.massive.client
%aimport quantdata.massive.taq

from datetime import date

from cdxcore.util import fmt_date
from cdxcore.verbose import Context

from quantdata.massive.client import MassiveClient
from quantdata.massive.taq import TAQMatchConfig, match_taq

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Raw trades and quotes

Use a small option ticker so the output is easy to inspect by eye.

In [7]:
ticker = "O:SPY240105P00470000"  # 5 trades on 2024-01-03 — easy to verify
d = date(2024, 1, 3)
client = MassiveClient()

with Context.all.write_t(f"Fetching trades for {ticker} on {fmt_date(d)}... ", end="") as tme:
    trades = client.fetch_trades(ticker, d)
    Context.all.write(f"done: {trades.height} trades. This took {tme}.", head=False)

with Context.all.write_t(f"Fetching quotes for {ticker} on {fmt_date(d)}... ", end="") as tme:
    quotes = client.fetch_quotes(ticker, d)
    Context.all.write(f"done: {quotes.height} quotes. This took {tme}.", head=False)

display(trades)
display(quotes.head(10))

00: Fetching trades for O:SPY240105P00470000 on 2024-01-03... done: 8895 trades. This took 1.1s.
00: Fetching quotes for O:SPY240105P00470000 on 2024-01-03... done: 85145 quotes. This took 5s.


ts_ns,price,size,trade_code
"datetime[ns, America/New_York]",f64,f64,u32
2024-01-03 09:30:00.424 EST,1.67,1.0,256
2024-01-03 09:30:00.472 EST,1.69,4.0,64
2024-01-03 09:30:00.576 EST,1.7,10.0,64
2024-01-03 09:30:00.617 EST,1.71,1.0,576
2024-01-03 09:30:01.480 EST,1.71,1.0,256
…,…,…,…
2024-01-03 16:11:20.628 EST,2.37,72.0,2624
2024-01-03 16:11:20.628 EST,2.37,72.0,2624
2024-01-03 16:12:51.108 EST,2.38,1.0,2624


ts_ns,bid_price,bid_size,ask_price,ask_size
"datetime[ns, America/New_York]",f64,f64,f64,f64
2024-01-03 09:30:00.183667200 EST,1.71,62.0,1.75,62.0
2024-01-03 09:30:00.217322240 EST,1.72,46.0,1.74,46.0
2024-01-03 09:30:00.383298048 EST,1.71,62.0,1.75,62.0
2024-01-03 09:30:00.400694528 EST,1.69,2.0,1.8,36.0
2024-01-03 09:30:00.419033088 EST,1.67,80.0,1.8,36.0
2024-01-03 09:30:00.424902144 EST,1.67,79.0,1.8,36.0
2024-01-03 09:30:00.460872704 EST,1.66,80.0,1.8,36.0
2024-01-03 09:30:00.470255104 EST,1.64,80.0,1.8,36.0
2024-01-03 09:30:00.489861888 EST,1.64,139.0,1.8,661.0


## TAQ matching

Run `match_taq` with the default config (1 s backward, 1 s forward) and inspect the full output.

In [8]:
config = TAQMatchConfig(
    backward_window_ns=250_000_000,  # 250 ms
    forward_window_ns=250_000_000,  # 250 ms
)
print(config)

matched = match_taq(ticker, d, config=config, verbose=Context.all)
display(matched)

bk=250ms,fw=250ms
00: Matching 8895 trades to 85145 quotes for O:SPY240105P00470000 on 2024-01-03.


ts_ns,price,size,trade_code,quote_ts_ns,bid_price,bid_size,ask_price,ask_size,quote_dt_ns
"datetime[ns, America/New_York]",f64,f64,u32,"datetime[ns, America/New_York]",f64,f64,f64,f64,i64
2024-01-03 09:30:00.424 EST,1.67,1.0,256,2024-01-03 09:30:00.424902144 EST,1.67,79.0,1.8,36.0,-902144
2024-01-03 09:30:00.472 EST,1.69,4.0,64,2024-01-03 09:30:00.470255104 EST,1.64,80.0,1.8,36.0,1744896
2024-01-03 09:30:00.576 EST,1.7,10.0,64,2024-01-03 09:30:00.573615872 EST,1.7,54.0,1.72,40.0,2384128
2024-01-03 09:30:00.617 EST,1.71,1.0,576,2024-01-03 09:30:00.573615872 EST,1.7,54.0,1.72,40.0,43384128
2024-01-03 09:30:01.480 EST,1.71,1.0,256,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…
2024-01-03 16:11:20.628 EST,2.37,72.0,2624,2024-01-03 16:11:20.520283392 EST,2.39,27.0,2.4,37.0,107716608
2024-01-03 16:11:20.628 EST,2.37,72.0,2624,2024-01-03 16:11:20.520283392 EST,2.39,27.0,2.4,37.0,107716608
2024-01-03 16:12:51.108 EST,2.38,1.0,2624,null,null,null,null,null,null


## Sanity checks

- Every matched row should have `|quote_dt_ns| <= max(backward_window_ns, forward_window_ns)`
- `quote_dt_ns > 0` means the quote was **before** the trade (pre-trade quote)
- `quote_dt_ns < 0` means the quote was **after** the trade (post-trade quote)

In [9]:
import polars as pl

n_matched = matched.filter(pl.col("quote_ts_ns").is_not_null()).height
n_unmatched = matched.filter(pl.col("quote_ts_ns").is_null()).height
print(f"Matched:   {n_matched}/{matched.height} trades")
print(f"Unmatched: {n_unmatched}/{matched.height} trades")

if n_matched > 0:
    dt_ms = matched.filter(pl.col("quote_dt_ns").is_not_null()).select(
        (pl.col("quote_dt_ns") / 1_000_000).alias("quote_dt_ms")
    )
    display(dt_ms.describe())

Matched:   6571/8895 trades
Unmatched: 2324/8895 trades


statistic,quote_dt_ms
str,f64
"""count""",6571.0
"""null_count""",0.0
"""mean""",-5.902448
"""std""",98.231753
"""min""",-249.684736
"""25%""",-56.763584
"""50%""",-1.195072
"""75%""",37.530688
"""max""",249.81408


## Try a tighter window

Verify that shrinking the window reduces matches.

In [ ]:
tight = TAQMatchConfig(
    backward_window_ns=10_000_000,  # 10 ms
    forward_window_ns=10_000_000,  # 10 ms
)
print(tight)

matched_tight = match_taq(ticker, d, config=tight, verbose=Context.all)
n_matched_tight = matched_tight.filter(pl.col("quote_ts_ns").is_not_null()).height
print(f"Matched with tight window: {n_matched_tight}/{matched_tight.height}")
display(matched_tight)

bk=10ms,fw=10ms
00: Matching 8895 trades to 85145 quotes for O:SPY240105P00470000 on 2024-01-03.
Matched with tight window: 1759/8895


ts_ns,price,size,trade_code,quote_ts_ns,bid_price,bid_size,ask_price,ask_size,quote_dt_ns
"datetime[ns, America/New_York]",f64,f64,u32,"datetime[ns, America/New_York]",f64,f64,f64,f64,i64
2024-01-03 09:30:00.424 EST,1.67,1.0,256,2024-01-03 09:30:00.424902144 EST,1.67,79.0,1.8,36.0,-902144
2024-01-03 09:30:00.472 EST,1.69,4.0,64,2024-01-03 09:30:00.470255104 EST,1.64,80.0,1.8,36.0,1744896
2024-01-03 09:30:00.576 EST,1.7,10.0,64,2024-01-03 09:30:00.573615872 EST,1.7,54.0,1.72,40.0,2384128
2024-01-03 09:30:00.617 EST,1.71,1.0,576,null,null,null,null,null,null
2024-01-03 09:30:01.480 EST,1.71,1.0,256,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…
2024-01-03 16:11:20.628 EST,2.37,72.0,2624,null,null,null,null,null,null
2024-01-03 16:11:20.628 EST,2.37,72.0,2624,null,null,null,null,null,null
2024-01-03 16:12:51.108 EST,2.38,1.0,2624,null,null,null,null,null,null
